# iPhone Downloads → Kaggle (no manually generated API token)

Run cells in order in Google Colab. First select **one ZIP** from your iPhone's Downloads folder. Then try Kaggle's **Google/browser OAuth login** rather than creating an API token. Finally the notebook publishes each `.ipynb` privately under **zeigleritaiml**.

**Important:** Kaggle's browser login may not complete in remote Colab, because the OAuth callback can expect the browser on the same machine as the CLI. If authentication fails, **stop**: the upload cannot proceed without a supported sign-in mechanism. No API token is requested or stored by this notebook.

In [ ]:
%pip -q install -U kaggle
from google.colab import files
from pathlib import Path
import zipfile, json, re, hashlib, shutil, subprocess
UPLOAD_ROOT = Path("/content/kaggle_iphone_upload")
UPLOAD_ROOT.mkdir(exist_ok=True)
print("Select ONE .zip file from On My iPhone → Downloads")
selected = files.upload()
zip_paths = [Path(name) for name in selected if name.lower().endswith(".zip")]
if len(zip_paths) != 1:
    raise ValueError("Please select one ZIP archive containing .ipynb notebooks.")
books = {}
with zipfile.ZipFile(zip_paths[0]) as archive:
    for member in archive.infolist():
        if member.is_dir() or not member.filename.lower().endswith(".ipynb"):
            continue
        filename = Path(member.filename).name
        payload = archive.read(member)
        doc = json.loads(payload)
        if doc.get("nbformat") != 4 or not isinstance(doc.get("cells"), list):
            raise ValueError(f"Not a supported Jupyter notebook: {filename}")
        if filename in books:
            raise ValueError(f"Duplicate filename in ZIP: {filename}")
        books[filename] = payload
if not books:
    raise ValueError("No notebooks found in ZIP.")
print(f"Found {len(books)} notebooks; ready for authentication.")


## Browser sign-in

The official Kaggle CLI supports `kaggle auth login --no-launch-browser` as an alternative to generating API tokens. Run the following cell; follow the sign-in URL it prints in Chrome. This flow is **not guaranteed to work inside hosted Colab**; if the process cannot receive the callback, it will not authenticate and uploading won't be possible from this session.

In [ ]:
# Browser-based OAuth. No token entry.
# A remote Colab session may be unable to receive the OAuth callback.
!kaggle auth login --no-launch-browser


In [ ]:
# Verify login before changing anything on Kaggle.
import subprocess
check = subprocess.run(["kaggle", "quota"], text=True, capture_output=True, timeout=30)
if check.returncode != 0:
    print(check.stderr[-700:] or check.stdout[-700:])
    raise RuntimeError("Kaggle login wasn't completed. Nothing was uploaded.")
print("Kaggle authentication check passed.")


In [ ]:
USERNAME = "zeigleritaiml"
print("Files selected:")
for name in sorted(books):
    print(" -", name)
if input(f"Publish {len(books)} PRIVATE notebooks to {USERNAME}? Type YES: ") != "YES":
    raise SystemExit("Cancelled.")
results = []
for index, (filename, payload) in enumerate(sorted(books.items()), 1):
    stem = Path(filename).stem
    base = re.sub(r"[^a-z0-9]+", "-", stem.lower()).strip("-")[:50]
    slug = base + "-" + hashlib.sha256(filename.encode()).hexdigest()[:8]
    folder = UPLOAD_ROOT / slug
    folder.mkdir(exist_ok=True)
    (folder / filename).write_bytes(payload)
    metadata = {
        "id": f"{USERNAME}/{slug}",
        "title": stem.replace("_", " ")[:100],
        "code_file": filename,
        "language": "python",
        "kernel_type": "notebook",
        "is_private": True,
        "enable_gpu": False,
        "enable_internet": False,
        "dataset_sources": [],
        "competition_sources": [],
        "kernel_sources": [],
        "model_sources": []
    }
    (folder / "kernel-metadata.json").write_text(json.dumps(metadata, indent=2))
    result = subprocess.run(
        ["kaggle", "kernels", "push", "-p", str(folder), "--no-run"],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else "FAILED"
    results.append((filename, status))
    print(f"{index}/{len(books)} {status}: {filename}")
    if result.returncode:
        print((result.stderr or result.stdout)[-500:])
print("Uploaded:", sum(status == "OK" for _, status in results), "of", len(results))
